In [1]:
import torch
from experiments.universal_encoder.universal_encoder_baseline import run_task_experiment
from experiments.utils import get_attribute_schema, get_text_embedder
from relbench.datasets import get_dataset
from relbench.tasks import get_task
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader

from redelex.data import make_pkey_fkey_graph, make_tensor_stats_dict
from redelex.nn.encoders import UniversalRowEncoder
from redelex.nn.encoders.universal_row_encoder import UniversalSAGEModel
from redelex.nn.train import get_node_train_table_input

%reload_ext autoreload
%autoreload 2

KeyboardInterrupt: 

In [ ]:
device = torch.device("cpu")  # torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
dataset_name = "rel-f1"
task_name = "driver-dnf"
cache_path = f".cache/{dataset_name}"

In [ ]:
dataset = get_dataset(dataset_name)
db = dataset.get_db(False)

Loading Database object from /home/jakub/.cache/relbench/rel-stack/db...
Done in 5.61 seconds.


In [ ]:
attribute_schema = get_attribute_schema(f"{cache_path}/attribute-schema.json", db)

In [ ]:
text_embedder_name = "glove"
text_embedder = get_text_embedder(text_embedder_name, torch.device("cpu"))

In [ ]:
# mini_table_dict = {
#     t: Table(
#         df=table.df[:10000],
#         fkey_col_to_pkey_table=table.fkey_col_to_pkey_table,
#         pkey_col=table.pkey_col,
#         time_col=table.time_col,
#     )
#     for t, table in db.table_dict.items()
# }
# mini_db = Database(table_dict=mini_table_dict)
# mini_db.reindex_pkeys_and_fkeys()

# data, col_stats_dict = make_pkey_fkey_graph(
#     mini_db,
#     col_to_stype_dict=attribute_schema,
#     text_embedder_cfg=TextEmbedderConfig(text_embedder, batch_size=252),
#     cache_dir=f"{cache_path}/materialized-mini",
# )

data, col_stats_dict = make_pkey_fkey_graph(
    db,
    col_to_stype_dict=attribute_schema,
    text_embedder=text_embedder,
    cache_dir=f"{cache_path}/materialized",
)

/home/jakub/Documents/CTU/ctu-relational/git/ctu-relational-py/.venv/lib/python3.12/site-packages/torch_frame/utils/io.py:113: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  warnings.warn(f"{warn_msg} Please use "
Embedding raw data in mini-batch: 100%|██████████| 5809/5809 [00:09<00:00, 630.61it/s]


In [ ]:
tensor_stats_dict = {}
name_embeddings_dict = {}
for tname in db.table_dict.keys():
    tensor_stats_dict[tname] = make_tensor_stats_dict(
        col_stats_dict=col_stats_dict[tname],
        col_names_dict=data[tname].tf.col_names_dict,
        text_embedder=text_embedder,
        device=device,
    )
    name_embeddings_dict[tname] = {
        s: text_embedder(s) for s in [tname, *col_stats_dict[tname].keys()]
    }
    data[tname].tensor_stats = tensor_stats_dict[tname]
    data[tname].name_embeddings = name_embeddings_dict[tname]

In [ ]:
def get_num_neighbors_dict(
    data: HeteroData,
    input_type: str,
    base_neighbors: int,
    num_layers: int,
    depth_factor: int,
) -> dict[str, list[int]]:
    num_neighbors_dict: dict[tuple[str, str, str], list[int]] = {
        e: [] for e in data.edge_types
    }
    layer_types = [input_type]
    edge_types = [e for e in data.edge_types if e[2] in layer_types]
    used_edge_types = []
    for n in range(num_layers):
        for e in data.edge_types:
            num_neighbors_dict[e].append(0)
        for edge_type in edge_types:
            src_type, _, dst_type = edge_type
            num_edges = len([e for e in edge_types if e[2] == dst_type])

            num_neighbors = max(
                1,
                base_neighbors // (depth_factor**n) // num_edges,
            )
            num_neighbors_dict[edge_type][-1] += num_neighbors

        used_edge_types.extend(edge_types)
        layer_types = list(set([src_type for src_type, _, dst_type in edge_types]))
        edge_types = [
            e
            for e in data.edge_types
            if e[2] in layer_types
            and all(
                [
                    not (e[0] == ue[0] and e[1] == ue[1] and e[2] == ue[2])
                    for ue in used_edge_types
                ]
            )
        ]
    return num_neighbors_dict


In [ ]:
task = get_task(dataset_name, task_name)
split = "train"
task_table = task.get_table(split)
num_neighbors = 128
num_layers = 2
is_temporal = True
batch_size = 512
num_workers = 0

in_edges_types: dict[str, list] = {}
for src_type, rel_type, dst_type in data.edge_types:
    if dst_type not in in_edges_types:
        in_edges_types[dst_type] = []
    in_edges_types[dst_type].append((src_type, rel_type, dst_type))

table_input = get_node_train_table_input(table=task_table, task=task)

num_neighbors_dict = get_num_neighbors_dict(
    data,
    input_type=table_input.nodes[0],
    base_neighbors=num_neighbors,
    num_layers=num_layers,
    depth_factor=2,
)

# for e, neighbors in num_neighbors_dict.items():
#     print(f"{e}: {neighbors}")

loader = NeighborLoader(
    data,
    num_neighbors=[num_neighbors // 2**i for i in range(num_layers)],  # num_neighbors_dict,
    time_attr="time" if is_temporal else None,
    input_nodes=table_input.nodes,
    input_time=table_input.time if is_temporal else None,
    transform=table_input.transform,
    batch_size=batch_size,
    temporal_strategy="uniform",
    shuffle=split == "train",
    num_workers=num_workers,
    persistent_workers=num_workers > 0,
)

In [ ]:
out_channels = 128
col_channels = 128
data_channels = col_channels // 2
type_channels = col_channels // 32
name_channels = col_channels * 3 // 8
stats_channels = col_channels * 3 // 32

encoder_heads = 8
encoder_layers = 2
encoder_dropout = 0

row_encoder = UniversalRowEncoder(
    out_channels=out_channels,
    text_embedder=text_embedder,
    type_channels=type_channels,
    name_channels=name_channels,
    stats_channels=stats_channels,
    data_channels=data_channels,
    encoder_heads=encoder_heads,
    encoder_layers=encoder_layers,
    encoder_dropout=encoder_dropout,
)


In [ ]:
def get_model_info(model: torch.nn.Module):
    param_size = 0
    total_params = 0
    trainable_params = 0
    buffer_size = 0
    total_buffers = 0

    for param in model.parameters():
        num_elements = param.nelement()
        total_params += num_elements
        if param.requires_grad:
            trainable_params += num_elements

        # Calculate bytes: (number of elements) * (size of one element)
        param_size += num_elements * param.element_size()

    for buffer in model.buffers():
        num_elements = buffer.nelement()
        total_buffers += num_elements
        buffer_size += num_elements * buffer.element_size()

    total_size_bytes = param_size + buffer_size
    total_size_mb = total_size_bytes / (1024**2)

    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "total_buffers": total_buffers,
        "size_bytes": total_size_bytes,
        "size_mb": total_size_mb,
    }


In [ ]:
upper_bound_samples = {
    n: [0] if n != table_input.nodes[0] else [batch_size] for n in data.node_types
}
for i in range(1, num_layers + 1):
    for n in data.node_types:
        upper_bound_samples[n].append(0)
        for e, neighbors in num_neighbors_dict.items():
            if e[0] == n:
                upper_bound_samples[n][i] += (
                    neighbors[i - 1] * upper_bound_samples[e[2]][i - 1]
                )


In [ ]:
model = UniversalSAGEModel(
    gnn_channels=128,
    col_channels=128,
    out_channels=1,
    text_embedder=text_embedder,
    node_types=data.node_types,
    edge_types=data.edge_types,
    gnn_layers=2,
    gnn_aggr="sum",
    head_norm="batch_norm",
)
model.to(device)

get_model_info(model)

{'total_params': 2043314,
 'trainable_params': 2043314,
 'total_buffers': 462,
 'size_bytes': 8175144,
 'size_mb': 7.796424865722656}

In [ ]:
from lightning.pytorch.utilities import disable_possible_user_warnings

# ignore all warnings that could be false positives
disable_possible_user_warnings()

run_task_experiment(
    config={
        "allow_gpu": True,
        "dataset_name": dataset_name,
        "task_name": task_name,
        # "mlflow_experiment": mlflow_experiment,
        # "mlflow_uri": mlflow_uri,
        "seed": 42,
        "text_embedder_name": text_embedder_name,
        "experiment_dir": "./experiments/universal_encoder/results",
        # training config
        "max_training_steps": 4000,
        "lr": 0.005 if dataset_name != "rel-trial" else 0.001,
        # sampling config
        "batch_size": 128,
        "num_neighbors": 16,
        # model config
        "col_channels": 128,
        "tabular_encoder_layers": 2,
        "tabular_encoder_heads": 8,
        "tabular_encoder_dropout": 0.0,
        "gnn_channels": 128,
        "gnn_layers": 2,
        "gnn_aggr": "mean",
        "head_norm": "batch_norm",
    },
    data=data,
    tensor_stats_dict=tensor_stats_dict,
    name_embeddings_dict=name_embeddings_dict,
    with_mlflow=False,
    with_ray=False,
)

Device: cuda


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type               | Params | Mode 
----------------------------------------------------------------
0 | model            | UniversalSAGEModel | 2.0 M  | train
1 | loss_fn          | BCEWithLogitsLoss  | 0      | train
2 | train_loss       | MeanMetric         | 0      | train
3 | best_tune_metric | MaxMetric          | 0      | train
----------------------------------------------------------------
2.0 M     Trainable params
0         Non-trainable params
2.0 M     Total params
8.173     Total estimated model params size (MB)
288       Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


Traceback (most recent call last):
  File "/home/jakub/Documents/CTU/ctu-relational/git/ctu-relational-py/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/call.py", line 48, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jakub/Documents/CTU/ctu-relational/git/ctu-relational-py/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/trainer.py", line 599, in _fit_impl
    self._run(model, ckpt_path=ckpt_path)
  File "/home/jakub/Documents/CTU/ctu-relational/git/ctu-relational-py/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/trainer.py", line 1012, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/home/jakub/Documents/CTU/ctu-relational/git/ctu-relational-py/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/trainer.py", line 1056, in _run_stage
    self.fit_loop.run()
  File "/home/jakub/Documents/CTU/ctu-relational/git/ctu-relational-py/.ve